## Análise — Perguntas do objetivo (Etapa 4.5)

**Fonte de dados:** tabelas gold (`workspace.gold.*`) do star schema CNO.

**Universo analítico** (definido em `docs/contexto-e-perguntas/definicoes.md`):
- Casas = áreas declaradas com `destinacao` em ('Residencial unifamiliar', 'Casa popular');
- somente área **Principal** (a casa em si) — `tipo_de_area = 'Principal'`;
- `categoria` em ('Existente', 'Obra Nova');
- métrica de tamanho: `metragem` (m²) — a silver já filtra `unidade_de_medida = m2`.

**Perguntas:**
- **P1.** Houve variação no tempo da área média das casas?
- **P2.** O tamanho populacional do município se relaciona com a área construída? E o da região geográfica imediata?
- **P3.** Distribuição das situações por porte de obra — obras menores têm mais chance de ficarem paralisadas/nulas?

**Alinhamento temporal P2:** população **no ano da obra** (join `fato_populacao.ano` = ano do início da obra). Obras em anos sem estimativa populacional (ex.: 1990, 2026+) ficam de fora do join.

In [0]:
%run ../ETL/notebooks/shared/_setup

In [0]:
from pyspark.sql import functions as F
from data_pipeline import adicionar_regiao

In [0]:
# Tabelas
FATO_OBRAS = 'workspace.gold.fato_obras'
FATO_POPULACAO = 'workspace.gold.fato_populacao'
DIM_DATA = 'workspace.gold.dim_data'
DIM_MUNICIPIO = 'workspace.gold.dim_municipio'
DIM_SITUACAO = 'workspace.gold.dim_situacao'
DIM_AREA = 'workspace.gold.dim_area'

# Universo analítico (definicoes.md)
DESTINACOES = ('Residencial unifamiliar', 'Casa popular')
CATEGORIAS = ('Existente', 'Obra Nova')

# P1: estados de maior porte a detalhar (ajustável conforme volume)
UF_FILTRO = ('SP', 'MG', 'RJ', 'BA')

# P2: faixas de população (município / região imediata)
FAIXA_POP_MUNICIPIO = '''
CASE
  WHEN populacao < 50000 THEN 'Ate 50 mil'
  WHEN populacao < 200000 THEN '50 a 200 mil'
  WHEN populacao < 1000000 THEN '200 mil a 1 milhao'
  ELSE 'Acima de 1 milhao'
END
'''
FAIXA_POP_REGIAO = '''
CASE
  WHEN populacao_regiao < 200000 THEN 'Ate 200 mil'
  WHEN populacao_regiao < 1000000 THEN '200 mil a 1 milhao'
  ELSE 'Acima de 1 milhao'
END
'''

# P3: faixas de metragem (70 m² = limiar de dispensa legal de registro no CNO)
FAIXA_METRAGEM = '''
CASE
  WHEN metragem < 70 THEN '0 - 70'
  WHEN metragem < 100 THEN '70 - 100'
  WHEN metragem < 500 THEN '100 - 500'
  WHEN metragem < 1000 THEN '500 - 1.000'
  ELSE '> 1.000'
END
'''

In [0]:
# v_municipio: dim_municipio enriquecida com a coluna `regiao`
spark.table(DIM_MUNICIPIO).transform(adicionar_regiao).createOrReplaceTempView('v_municipio')

# v_casas: universo analítico (casas) pronto para as consultas das perguntas
df_casas = (
    spark.table(FATO_OBRAS)
    .join(
        spark.table(DIM_DATA).select('sk_data', 'ano'),
        on=F.col('sk_data_inicio') == F.col('sk_data'),
        how='inner',
    )
    .join(
        spark.table(DIM_AREA).select('sk_area', 'destinacao', 'tipo_de_area', 'categoria'),
        on='sk_area',
        how='inner',
    )
    .join(
        spark.table(DIM_SITUACAO).select('sk_situacao', 'descricao'),
        on='sk_situacao',
        how='inner',
    )
    .join(
        spark.table('v_municipio').select(
            'sk_municipio',
            'codigo_municipio',
            'sigla_uf',
            'regiao',
            'codigo_regiao_geografica_imediata',
        ),
        on='sk_municipio',
        how='inner',
    )
    .filter(F.col('destinacao').isin(*DESTINACOES))
    .filter(F.col('tipo_de_area') == 'Principal')
    .filter(F.col('categoria').isin(*CATEGORIAS))
    .select(
        'cno',
        'metragem',
        'ano',
        'sigla_uf',
        'regiao',
        'codigo_municipio',
        'codigo_regiao_geografica_imediata',
        F.col('descricao').alias('situacao_descricao'),
    )
)
df_casas.createOrReplaceTempView('v_casas')

print(f'Universo analítico (casas): {df_casas.count():,} linhas')

## P1 — Variação temporal da metragem média

**Como responder:** média (e mediana) da `metragem` das casas por ano/década de início da obra, em três níveis:
1. **Brasil** (todos os registros);
2. **Por região** (Norte, Nordeste, Centro-Oeste, Sudeste, Sul — derivada da UF);
3. **Por estado** (principais UFs — ajustável na constante `UF_FILTRO`).

**Discussão esperada:** há tendência de aumento/redução? A amostra por ano é suficiente nos anos iniciais? Considerar suavizar por década se houver poucos registros.

In [0]:
# P1.1a - Brasil: metragem media por ano
print('P1.1a - Brasil por ano')
display(spark.sql('''
SELECT
  ano,
  COUNT(*) AS qtd_casas,
  ROUND(AVG(metragem), 2) AS metragem_media,
  ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY metragem), 2) AS metragem_mediana
FROM v_casas
GROUP BY ano
ORDER BY ano
'''))

# P1.1b - Brasil: metragem media por decada (suaviza anos com amostra pequena)
print('P1.1b - Brasil por decada')
display(spark.sql('''
SELECT
  FLOOR(ano / 10) * 10 AS decada,
  COUNT(*) AS qtd_casas,
  ROUND(AVG(metragem), 2) AS metragem_media
FROM v_casas
GROUP BY decada
ORDER BY decada
'''))

In [0]:
# P1.2 - Por regiao (Norte, Nordeste, Centro-Oeste, Sudeste, Sul)
print('P1.2 - Metragem media por regiao e ano')
display(spark.sql('''
SELECT
  regiao,
  ano,
  COUNT(*) AS qtd_casas,
  ROUND(AVG(metragem), 2) AS metragem_media
FROM v_casas
WHERE regiao IS NOT NULL
GROUP BY regiao, ano
ORDER BY regiao, ano
'''))

In [0]:
# P1.3 - Por estado (principais UFs)
print(f'P1.3 - Metragem media por ano para: {", ".join(UF_FILTRO)}')
ufs = ', '.join(f"'{u}'" for u in UF_FILTRO)
display(spark.sql(f'''
SELECT
  sigla_uf,
  ano,
  COUNT(*) AS qtd_casas,
  ROUND(AVG(metragem), 2) AS metragem_media
FROM v_casas
WHERE sigla_uf IN ({ufs})
GROUP BY sigla_uf, ano
ORDER BY sigla_uf, ano
'''))

## P2 — População × área construída

**Como responder:** comparar a `metragem` média das casas por **faixa de população**, usando a população **no ano da obra** (`fato_populacao.ano` = ano de início):
1. população do **município** da obra;
2. população da **região geográfica imediata** (soma da população dos municípios da região).

**Discussão esperada:** municípios/regiões mais populosos têm casas menores (adensamento, custo do terreno)? Analisar por faixas — cuidado com falácia ecológica.

In [0]:
# P2.1 - Faixas de populacao do municipio (ano da obra)
print('P2.1 - Metragem media por faixa de populacao do municipio')
display(spark.sql(f'''
SELECT
  {FAIXA_POP_MUNICIPIO} AS faixa_populacao,
  COUNT(*) AS qtd_casas,
  ROUND(AVG(c.metragem), 2) AS metragem_media
FROM v_casas c
JOIN {FATO_POPULACAO} p
  ON c.codigo_municipio = p.codigo_municipio AND c.ano = p.ano
GROUP BY faixa_populacao
ORDER BY MIN(p.populacao)
'''))

In [0]:
# P2.2 - Faixas de populacao da regiao geografica imediata (ano da obra)
print('P2.2 - Metragem media por faixa de populacao da regiao imediata')
display(spark.sql(f'''
WITH pop_regiao AS (
  SELECT
    p.ano,
    m.codigo_regiao_geografica_imediata,
    SUM(p.populacao) AS populacao_regiao
  FROM {FATO_POPULACAO} p
  JOIN {DIM_MUNICIPIO} m ON p.sk_municipio = m.sk_municipio
  GROUP BY p.ano, m.codigo_regiao_geografica_imediata
)
SELECT
  {FAIXA_POP_REGIAO} AS faixa_populacao_regiao,
  COUNT(*) AS qtd_casas,
  ROUND(AVG(c.metragem), 2) AS metragem_media
FROM v_casas c
JOIN pop_regiao r
  ON c.codigo_regiao_geografica_imediata = r.codigo_regiao_geografica_imediata
 AND c.ano = r.ano
GROUP BY faixa_populacao_regiao
ORDER BY MIN(r.populacao_regiao)
'''))

## P3 — Situação da obra × porte

**Como responder:**
1. distribuição da `metragem` por situação da obra (`dim_situacao.descricao`);
2. taxa de obras `PARALISADA`/`NULA` por faixa de metragem (limiar de 70 m² = dispensa legal de registro).

**Discussão esperada:** obras menores estão mais sujeitas a paralisação/nulidade? A distribuição por situação é coerente com o ciclo de vida das obras?

In [0]:
# P3.1 - Distribuicao da metragem por situacao
print('P3.1 - Metragem por situacao da obra')
display(spark.sql('''
SELECT
  situacao_descricao,
  COUNT(*) AS qtd_casas,
  ROUND(AVG(metragem), 2) AS metragem_media,
  ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY metragem), 2) AS metragem_mediana
FROM v_casas
GROUP BY situacao_descricao
ORDER BY qtd_casas DESC
'''))

In [0]:
# P3.2 - Taxa de PARALISADA/NULA por faixa de metragem
print('P3.2 - Obras paralisadas/nulas por faixa de metragem')
display(spark.sql(f'''
SELECT
  {FAIXA_METRAGEM} AS faixa_metragem,
  COUNT(*) AS qtd_casas,
  SUM(CASE WHEN situacao_descricao IN ('PARALISADA', 'NULA') THEN 1 ELSE 0 END) AS paralisada_nula,
  ROUND(
    100 * SUM(CASE WHEN situacao_descricao IN ('PARALISADA', 'NULA') THEN 1 ELSE 0 END) / COUNT(*),
    2
  ) AS pct_paralisada_nula
FROM v_casas
GROUP BY faixa_metragem
ORDER BY MIN(metragem)
'''))

## Discussão geral

Consolidar as respostas de P1, P2 e P3 conectando os números ao problema original (“Panorama do tamanho das casas populares e unifamiliares conforme o CNO”).

Considerar as limitações registradas em `docs/contexto-e-perguntas/definicoes.md`:
- sub-representação de casas ≤ 70 m² (dispensa legal de registro no CNO);
- cadastro com finalidade tributária (obras informais podem faltar);
- alinhamento temporal da população (ano da obra) — obras em 1990/2026+ ficam sem pareamento.